# Model Calibration

Testing whether predicted probability actually corresponds to observed conversion rates.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss
)

df = pd.read_csv(
    "../data/processed/model_data.csv"
)

X = df.drop(columns=["y"])
y = df["y"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [ ]:
# Recreate the XGBoost pipeline from previous notebooks
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()
numerical_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

preprocessor = ColumnTransformer([
    ("numeric", Pipeline([("imputer", SimpleImputer(strategy="median"))]), numerical_features),
    ("categorical", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("encoder", OneHotEncoder(handle_unknown="ignore"))]), categorical_features)
])

xgb_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBClassifier(n_estimators=400, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, eval_metric="logloss", random_state=42))
])

xgb_model.fit(X_train, y_train)

xgb_probability = (
    xgb_model
    .predict_proba(X_test)[:, 1]
)

In [ ]:
print(
    "ROC-AUC:",
    roc_auc_score(
        y_test,
        xgb_probability
    )
)

print(
    "PR-AUC:",
    average_precision_score(
        y_test,
        xgb_probability
    )
)

print(
    "Brier Score:",
    brier_score_loss(
        y_test,
        xgb_probability
    )
)

print(
    "Log Loss:",
    log_loss(
        y_test,
        xgb_probability
    )
)

In [ ]:
from sklearn.calibration import calibration_curve

prob_true, prob_pred = calibration_curve(
    y_test,
    xgb_probability,
    n_bins=10,
    strategy="quantile"
)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))

plt.plot(
    prob_pred,
    prob_true,
    marker="o",
    label="XGBoost"
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Perfect calibration"
)

plt.xlabel("Mean predicted probability")
plt.ylabel("Observed conversion rate")
plt.title("Model Calibration")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.calibration import CalibratedClassifierCV

calibrated_xgb = CalibratedClassifierCV(
    xgb_model,
    method="sigmoid",
    cv=5
)

calibrated_xgb.fit(
    X_train,
    y_train
)

calibrated_probability = (
    calibrated_xgb
    .predict_proba(X_test)[:, 1]
)

In [ ]:
comparison = pd.DataFrame({
    "Metric": [
        "ROC-AUC",
        "PR-AUC",
        "Brier Score",
        "Log Loss"
    ],
    "XGBoost": [
        roc_auc_score(y_test, xgb_probability),
        average_precision_score(y_test, xgb_probability),
        brier_score_loss(y_test, xgb_probability),
        log_loss(y_test, xgb_probability)
    ],
    "Calibrated XGBoost": [
        roc_auc_score(y_test, calibrated_probability),
        average_precision_score(y_test, calibrated_probability),
        brier_score_loss(y_test, calibrated_probability),
        log_loss(y_test, calibrated_probability)
    ]
})

comparison

In [ ]:
cal_true, cal_pred = calibration_curve(
    y_test,
    calibrated_probability,
    n_bins=10,
    strategy="quantile"
)

plt.figure(figsize=(8, 6))

plt.plot(
    prob_pred,
    prob_true,
    marker="o",
    label="XGBoost"
)

plt.plot(
    cal_pred,
    cal_true,
    marker="o",
    label="Calibrated XGBoost"
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Perfect calibration"
)

plt.xlabel("Mean predicted probability")
plt.ylabel("Observed conversion rate")
plt.title("Calibration Comparison")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# From now on, our final model probability should be:
final_probability = calibrated_probability